# 🌾 PRAG — Entraînement GPU sur Kaggle
> **Prédiction des meilleures périodes agricoles — Côte d'Ivoire**
>
> Ce notebook entraîne des réseaux de neurones sur GPU (T4 Kaggle) pour :
> - **Classification** : `periode_semis_optimale` (0/1)
> - **Régression** : `rendement_tonnes_ha`
>
> Puis compare avec les modèles CPU et sélectionne le meilleur global.

---
**Instructions :**
1. Uploadez `train.csv` et `test.csv` dans Kaggle > Data > Upload Dataset
2. Activez le GPU : Settings > Accelerator > GPU T4 x2
3. Run All


In [ ]:
# ── Installation des dépendances ─────────────────────────────────────────
!pip install lightgbm xgboost torch scikit-learn -q

In [ ]:
import json, time, os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from datetime import datetime, timezone
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device utilisé : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# ── Chargement des données ────────────────────────────────────────────────
# Adaptez le chemin si votre dataset est dans /kaggle/input/
DATA_PATH = '/kaggle/input/prag-agripredict/'  # ou './' si upload direct

try:
    train = pd.read_csv(DATA_PATH + 'train.csv')
    test  = pd.read_csv(DATA_PATH + 'test.csv')
except FileNotFoundError:
    # Fallback : fichiers dans le répertoire courant
    train = pd.read_csv('train.csv')
    test  = pd.read_csv('test.csv')

print(f'Train : {train.shape} | Test : {test.shape}')
print(f"Équilibre classes : {train['periode_semis_optimale'].value_counts().to_dict()}")

In [ ]:
# ── Prétraitement ────────────────────────────────────────────────────────
CAT_COLS     = ['region', 'culture', 'type_sol', 'saison']
FEATURE_COLS = ['annee','mois','trimestre','jour_annee','sin_mois','cos_mois',
                'pluviometrie_mm','temperature_celsius','humidite_pct',
                'region','culture','type_sol','saison']

X_all = pd.concat([train[FEATURE_COLS], test[FEATURE_COLS]], axis=0)
X_all_enc = pd.get_dummies(X_all, columns=CAT_COLS, drop_first=False)

n_train   = len(train)
X_train   = X_all_enc.iloc[:n_train].values.astype(np.float32)
X_test    = X_all_enc.iloc[n_train:].values.astype(np.float32)

y_clf_train = train['periode_semis_optimale'].values.astype(np.float32)
y_clf_test  = test['periode_semis_optimale'].values.astype(np.float32)
y_reg_train = train['rendement_tonnes_ha'].values.astype(np.float32)
y_reg_test  = test['rendement_tonnes_ha'].values.astype(np.float32)

# Normalisation pour les réseaux de neurones
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled  = scaler.transform(X_test).astype(np.float32)

N_FEATURES = X_train.shape[1]
print(f'Features : {N_FEATURES} colonnes après encodage')

In [ ]:
# ── Architecture du réseau de neurones ───────────────────────────────────
class PRAGNet(nn.Module):
    """Réseau de neurones générique pour PRAG.
    
    task='clf' → sortie sigmoïde (classification binaire)
    task='reg' → sortie linéaire (régression)
    """
    def __init__(self, n_features: int, task: str = 'clf', dropout: float = 0.3):
        super().__init__()
        self.task = task
        self.net = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        if task == 'clf':
            self.out = nn.Sigmoid()
        else:
            self.out = nn.Identity()

    def forward(self, x):
        return self.out(self.net(x)).squeeze(-1)


def train_nn(X_tr, y_tr, X_te, y_te, task='clf', epochs=50, lr=1e-3, batch_size=256):
    """Entraîne PRAGNet et retourne les métriques + temps d'inférence."""
    model = PRAGNet(X_tr.shape[1], task=task).to(DEVICE)

    # Tensors
    Xt = torch.tensor(X_tr).to(DEVICE)
    yt = torch.tensor(y_tr).to(DEVICE)
    Xv = torch.tensor(X_te).to(DEVICE)
    yv = torch.tensor(y_te).to(DEVICE)

    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    criterion = nn.BCELoss() if task == 'clf' else nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    t0 = time.perf_counter()
    model.train()
    for epoch in range(epochs):
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs} — loss={loss.item():.4f}')
    train_time = time.perf_counter() - t0

    # Inférence
    model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        preds = model(Xv).cpu().numpy()
    infer_ms = (time.perf_counter() - t1) * 1000 / len(y_te)

    if task == 'clf':
        y_pred = (preds >= 0.5).astype(int)
        metrics = {
            'f1_score': round(f1_score(y_te, y_pred), 4),
            'accuracy': round(accuracy_score(y_te, y_pred), 4),
            'roc_auc': round(roc_auc_score(y_te, preds), 4),
            'train_time_sec': round(train_time, 2),
            'inference_ms_per_obs': round(infer_ms, 5)
        }
    else:
        rmse = float(np.sqrt(mean_squared_error(y_te, preds)))
        metrics = {
            'rmse': round(rmse, 4),
            'r2_score': round(r2_score(y_te, preds), 4),
            'train_time_sec': round(train_time, 2),
            'inference_ms_per_obs': round(infer_ms, 5)
        }

    return model, metrics, preds

In [ ]:
# ── Entraînement Classification GPU ──────────────────────────────────────
print('='*60)
print('CLASSIFICATION — periode_semis_optimale')
print('='*60)

clf_history = []

# 1) MLP standard
print('\n▶ MLP (GPU)')
mlp_clf, mlp_clf_metrics, _ = train_nn(
    X_train_scaled, y_clf_train,
    X_test_scaled, y_clf_test,
    task='clf', epochs=60, lr=5e-4
)
print(f"  F1={mlp_clf_metrics['f1_score']}  AUC={mlp_clf_metrics['roc_auc']}  "
      f"Inférence={mlp_clf_metrics['inference_ms_per_obs']:.4f}ms")
clf_history.append({'model':'MLP_GPU', 'device':DEVICE, 'metrics': mlp_clf_metrics})

# 2) MLP profond (plus de couches)
class DeepPRAGNet(nn.Module):
    def __init__(self, n_features, task='clf'):
        super().__init__()
        self.task = task
        self.net = nn.Sequential(
            nn.Linear(n_features, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.out = nn.Sigmoid() if task == 'clf' else nn.Identity()
    def forward(self, x):
        return self.out(self.net(x)).squeeze(-1)

print('\n▶ DeepMLP (GPU)')
deep_clf = DeepPRAGNet(N_FEATURES, 'clf').to(DEVICE)
Xt = torch.tensor(X_train_scaled).to(DEVICE)
yt = torch.tensor(y_clf_train).to(DEVICE)
Xv = torch.tensor(X_test_scaled).to(DEVICE)
loader = DataLoader(TensorDataset(Xt, yt), batch_size=256, shuffle=True)
opt = optim.AdamW(deep_clf.parameters(), lr=3e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=80)
t0 = time.perf_counter()
deep_clf.train()
for ep in range(80):
    for Xb, yb in loader:
        opt.zero_grad()
        nn.BCELoss()(deep_clf(Xb), yb).backward()
        opt.step()
    sched.step()
    if (ep+1) % 20 == 0: print(f'   Epoch {ep+1}/80')
train_time_deep = time.perf_counter() - t0
deep_clf.eval()
t1 = time.perf_counter()
with torch.no_grad(): preds_deep = deep_clf(Xv).cpu().numpy()
infer_deep = (time.perf_counter() - t1) * 1000 / len(y_clf_test)
f1_deep = f1_score(y_clf_test, (preds_deep >= 0.5).astype(int))
auc_deep = roc_auc_score(y_clf_test, preds_deep)
deep_clf_metrics = {'f1_score': round(f1_deep,4), 'roc_auc': round(auc_deep,4),
                    'train_time_sec': round(train_time_deep,2),
                    'inference_ms_per_obs': round(infer_deep,5)}
print(f"  F1={deep_clf_metrics['f1_score']}  AUC={deep_clf_metrics['roc_auc']}")
clf_history.append({'model':'DeepMLP_GPU', 'device':DEVICE, 'metrics': deep_clf_metrics})

In [ ]:
# ── Entraînement Régression GPU ───────────────────────────────────────────
print('='*60)
print('RÉGRESSION — rendement_tonnes_ha')
print('='*60)

reg_history = []

print('\n▶ MLP Régression (GPU)')
mlp_reg, mlp_reg_metrics, _ = train_nn(
    X_train_scaled, y_reg_train,
    X_test_scaled, y_reg_test,
    task='reg', epochs=80, lr=5e-4
)
print(f"  RMSE={mlp_reg_metrics['rmse']}  R²={mlp_reg_metrics['r2_score']}")
reg_history.append({'model':'MLP_GPU', 'device':DEVICE, 'metrics': mlp_reg_metrics})

print('\n▶ DeepMLP Régression (GPU)')
deep_reg = DeepPRAGNet(N_FEATURES, 'reg').to(DEVICE)
yt_reg = torch.tensor(y_reg_train).to(DEVICE)
loader_reg = DataLoader(TensorDataset(Xt, yt_reg), batch_size=256, shuffle=True)
opt_reg = optim.AdamW(deep_reg.parameters(), lr=3e-4, weight_decay=1e-4)
sched_reg = optim.lr_scheduler.CosineAnnealingLR(opt_reg, T_max=80)
t0 = time.perf_counter()
deep_reg.train()
for ep in range(80):
    for Xb, yb in loader_reg:
        opt_reg.zero_grad()
        nn.MSELoss()(deep_reg(Xb), yb).backward()
        opt_reg.step()
    sched_reg.step()
    if (ep+1) % 20 == 0: print(f'   Epoch {ep+1}/80')
train_time_deep_reg = time.perf_counter() - t0
deep_reg.eval()
t1 = time.perf_counter()
with torch.no_grad(): preds_reg_deep = deep_reg(Xv).cpu().numpy()
infer_deep_reg = (time.perf_counter() - t1) * 1000 / len(y_reg_test)
rmse_deep = float(np.sqrt(mean_squared_error(y_reg_test, preds_reg_deep)))
r2_deep = r2_score(y_reg_test, preds_reg_deep)
deep_reg_metrics = {'rmse': round(rmse_deep,4), 'r2_score': round(r2_deep,4),
                    'train_time_sec': round(train_time_deep_reg,2),
                    'inference_ms_per_obs': round(infer_deep_reg,5)}
print(f"  RMSE={deep_reg_metrics['rmse']}  R²={deep_reg_metrics['r2_score']}")
reg_history.append({'model':'DeepMLP_GPU', 'device':DEVICE, 'metrics': deep_reg_metrics})

In [ ]:
# ── Résumé & export de l'historique GPU ─────────────────────────────────
print('\n' + '='*60)
print('RÉSUMÉ — MODÈLES GPU')
print('='*60)

print('\n📊 Classification :')
for r in clf_history:
    print(f"  {r['model']:20s} | F1={r['metrics']['f1_score']}  "
          f"AUC={r['metrics'].get('roc_auc','N/A')}  "
          f"Inférence={r['metrics']['inference_ms_per_obs']:.4f}ms")

print('\n📊 Régression :')
for r in reg_history:
    print(f"  {r['model']:20s} | RMSE={r['metrics']['rmse']}  "
          f"R²={r['metrics']['r2_score']}  "
          f"Inférence={r['metrics']['inference_ms_per_obs']:.4f}ms")

# Export de l'historique GPU au format JSON
gpu_history = {
    'session': datetime.now(timezone.utc).isoformat(),
    'device': DEVICE,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU',
    'classification_runs': clf_history,
    'regression_runs': reg_history
}

with open('prag_gpu_history.json', 'w', encoding='utf-8') as f:
    json.dump(gpu_history, f, indent=2, ensure_ascii=False)

print('\n✅ Historique GPU exporté → prag_gpu_history.json')
print('   Téléchargez ce fichier et placez-le dans logs/ de votre projet PRAG')

# Sauvegarder les modèles PyTorch
best_clf_gpu = max(clf_history, key=lambda r: r['metrics']['f1_score'])
best_reg_gpu = max(reg_history, key=lambda r: -r['metrics']['rmse'])

if best_clf_gpu['model'] == 'MLP_GPU':
    torch.save(mlp_clf.state_dict(), 'prag_clf_mlp_gpu.pt')
else:
    torch.save(deep_clf.state_dict(), 'prag_clf_deepmlp_gpu.pt')

if best_reg_gpu['model'] == 'MLP_GPU':
    torch.save(mlp_reg.state_dict(), 'prag_reg_mlp_gpu.pt')
else:
    torch.save(deep_reg.state_dict(), 'prag_reg_deepmlp_gpu.pt')

print(f'\n✅ Meilleur CLF GPU : {best_clf_gpu["model"]} | F1={best_clf_gpu["metrics"]["f1_score"]}')
print(f'✅ Meilleur REG GPU : {best_reg_gpu["model"]} | RMSE={best_reg_gpu["metrics"]["rmse"]}')